[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part3_spatial_ai/ch18_computational_structure/01_spatial_ai_system.ipynb)

# 第18章: 空間AIの計算構造

本ノートブックでは、SLAMシステムの計算パイプライン設計とシステムレベルの考慮事項について学びます。

## 学習内容
1. **タスクグラフ / 計算DAG**: SLAMパイプラインのモジュール間依存関係
2. **レイテンシ分析**: 各処理段階の計算時間と全体のスループット
3. **階層的空間認識**: マルチスケールな空間理解のアーキテクチャ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
np.random.seed(42)

## 1. SLAMパイプラインの計算DAG

現代のSLAMシステムは、複数のモジュールが依存関係を持つDAG（有向非巡回グラフ）として構成されます。フロントエンド（リアルタイム処理）とバックエンド（最適化処理）は異なる周期で動作し、並列実行される場合もあります。

In [ ]:
# --- SLAM Pipeline Computational DAG ---

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 8.5)
ax.set_aspect('equal')
ax.axis('off')

# Define nodes: (name, x, y, color, group)
nodes = {
    'sensor':       ('Sensor\nInput', 0.5, 4, '#E8E8E8', 'input'),
    'preprocess':   ('Image\nPreprocess', 2.5, 6, '#FFD700', 'frontend'),
    'imu_preint':   ('IMU\nPreintegration', 2.5, 2, '#FFD700', 'frontend'),
    'features':     ('Feature\nExtraction', 4.5, 6, '#FF8C00', 'frontend'),
    'tracking':     ('Visual\nTracking', 6.5, 6, '#FF8C00', 'frontend'),
    'depth':        ('Depth\nEstimation', 4.5, 4, '#FF8C00', 'frontend'),
    'semantic':     ('Semantic\nSegmentation', 4.5, 8, '#87CEEB', 'DNN'),
    'odom':         ('Visual-Inertial\nOdometry', 6.5, 3, '#90EE90', 'frontend'),
    'keyframe':     ('Keyframe\nDecision', 8, 5.5, '#DDA0DD', 'backend'),
    'local_ba':     ('Local Bundle\nAdjustment', 8, 3.5, '#DDA0DD', 'backend'),
    'loop':         ('Loop\nDetection', 9.5, 7, '#F08080', 'backend'),
    'global_opt':   ('Global\nOptimization', 9.5, 4.5, '#F08080', 'backend'),
    'map':          ('3D Map\nUpdate', 9.5, 2, '#98FB98', 'output'),
}

# Draw nodes
for key, (label, x, y, color, group) in nodes.items():
    box = FancyBboxPatch((x - 0.8, y - 0.5), 1.6, 1.0,
                         boxstyle="round,pad=0.1", facecolor=color,
                         edgecolor='black', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=8, fontweight='bold')

# Define edges
edges = [
    ('sensor', 'preprocess'), ('sensor', 'imu_preint'),
    ('preprocess', 'features'), ('preprocess', 'depth'), ('preprocess', 'semantic'),
    ('features', 'tracking'),
    ('tracking', 'odom'), ('imu_preint', 'odom'), ('depth', 'odom'),
    ('tracking', 'keyframe'), ('odom', 'keyframe'),
    ('keyframe', 'local_ba'), ('odom', 'local_ba'),
    ('keyframe', 'loop'), ('semantic', 'loop'),
    ('loop', 'global_opt'), ('local_ba', 'global_opt'),
    ('global_opt', 'map'), ('local_ba', 'map'),
]

for src, dst in edges:
    sx, sy = nodes[src][1], nodes[src][2]
    dx, dy = nodes[dst][1], nodes[dst][2]
    ax.annotate('', xy=(dx - 0.7 * np.sign(dx - sx) if abs(dx - sx) > 1.5 else dx,
                        dy - 0.4 * np.sign(dy - sy) if abs(dy - sy) > 0.8 else dy),
                xytext=(sx + 0.7 * np.sign(dx - sx) if abs(dx - sx) > 1.5 else sx,
                        sy + 0.4 * np.sign(dy - sy) if abs(dy - sy) > 0.8 else sy),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, connectionstyle='arc3,rad=0.1'))

# Legend
legend_items = [
    mpatches.Patch(color='#FFD700', label='Preprocessing'),
    mpatches.Patch(color='#FF8C00', label='Frontend (real-time)'),
    mpatches.Patch(color='#87CEEB', label='DNN Module'),
    mpatches.Patch(color='#DDA0DD', label='Backend (optimization)'),
    mpatches.Patch(color='#F08080', label='Global Backend'),
]
ax.legend(handles=legend_items, loc='lower left', fontsize=10, framealpha=0.9)

ax.set_title('Spatial AI System - Computational DAG', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('slam_dag.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. レイテンシ分析

各モジュールの処理時間を分析し、ボトルネックを特定します。リアルタイム性の要件（例：30fps = 33ms以内）を満たすためには、クリティカルパス上の処理時間が重要です。

In [ ]:
# --- Latency Analysis of SLAM Pipeline ---

# Typical processing times (in milliseconds)
modules = {
    'Image Preprocess':       {'time_ms': 2, 'category': 'frontend', 'thread': 'main'},
    'Feature Extraction':     {'time_ms': 8, 'category': 'frontend', 'thread': 'main'},
    'Feature Matching':       {'time_ms': 5, 'category': 'frontend', 'thread': 'main'},
    'Depth Estimation (DNN)': {'time_ms': 15, 'category': 'DNN', 'thread': 'GPU'},
    'Semantic Seg (DNN)':     {'time_ms': 20, 'category': 'DNN', 'thread': 'GPU'},
    'Visual Odometry':        {'time_ms': 6, 'category': 'frontend', 'thread': 'main'},
    'IMU Preintegration':     {'time_ms': 1, 'category': 'frontend', 'thread': 'main'},
    'Keyframe Selection':     {'time_ms': 1, 'category': 'backend', 'thread': 'main'},
    'Local BA':               {'time_ms': 30, 'category': 'backend', 'thread': 'backend'},
    'Loop Detection':         {'time_ms': 15, 'category': 'backend', 'thread': 'backend'},
    'Global Optimization':    {'time_ms': 100, 'category': 'backend', 'thread': 'backend'},
    'Map Update':             {'time_ms': 5, 'category': 'backend', 'thread': 'backend'},
}

# Critical path (sequential frontend)
frontend_modules = ['Image Preprocess', 'Feature Extraction', 'Feature Matching', 
                     'Visual Odometry', 'IMU Preintegration', 'Keyframe Selection']
frontend_time = sum(modules[m]['time_ms'] for m in frontend_modules)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of individual module times
names = list(modules.keys())
times = [modules[m]['time_ms'] for m in names]
colors_map = {'frontend': '#FF8C00', 'DNN': '#87CEEB', 'backend': '#DDA0DD'}
colors = [colors_map[modules[m]['category']] for m in names]

bars = axes[0].barh(range(len(names)), times, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names, fontsize=9)
axes[0].set_xlabel('Processing Time (ms)', fontsize=12)
axes[0].set_title('Per-Module Latency', fontsize=14, fontweight='bold')
axes[0].axvline(x=33.3, color='red', linestyle='--', linewidth=2, label='30fps budget (33ms)')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='x')

# Add time labels
for bar, time in zip(bars, times):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{time}ms', va='center', fontsize=9)

# Timeline / Gantt chart showing parallel execution
ax2 = axes[1]
thread_y = {'main': 3, 'GPU': 2, 'backend': 1}
thread_labels = {'main': 'Main Thread', 'GPU': 'GPU Thread', 'backend': 'Backend Thread'}

# Simulate timeline
timeline = []
thread_time = {t: 0 for t in thread_y}

# Frontend runs sequentially on main thread
for name in frontend_modules:
    m = modules[name]
    start = thread_time['main']
    end = start + m['time_ms']
    timeline.append((name, 'main', start, end, m['category']))
    thread_time['main'] = end

# GPU tasks run in parallel
gpu_start = modules['Image Preprocess']['time_ms']  # after preprocess
for name in ['Depth Estimation (DNN)', 'Semantic Seg (DNN)']:
    m = modules[name]
    start = max(gpu_start, thread_time['GPU'])
    end = start + m['time_ms']
    timeline.append((name, 'GPU', start, end, m['category']))
    thread_time['GPU'] = end

# Backend runs asynchronously
backend_start = frontend_time
for name in ['Local BA', 'Loop Detection', 'Global Optimization', 'Map Update']:
    m = modules[name]
    start = max(backend_start, thread_time['backend'])
    end = start + m['time_ms']
    timeline.append((name, 'backend', start, end, m['category']))
    thread_time['backend'] = end

for name, thread, start, end, cat in timeline:
    y = thread_y[thread]
    color = colors_map[cat]
    ax2.barh(y, end - start, left=start, height=0.6, color=color, 
             edgecolor='black', linewidth=0.5)
    if end - start > 8:
        ax2.text((start + end) / 2, y, name.split('(')[0].strip(), 
                ha='center', va='center', fontsize=7, fontweight='bold')

ax2.set_yticks(list(thread_y.values()))
ax2.set_yticklabels([thread_labels[t] for t in thread_y], fontsize=10)
ax2.set_xlabel('Time (ms)', fontsize=12)
ax2.set_title('Pipeline Timeline (Parallel Execution)', fontsize=14, fontweight='bold')
ax2.axvline(x=33.3, color='red', linestyle='--', linewidth=2, label='30fps deadline')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('latency_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"フロントエンド クリティカルパス: {frontend_time}ms")
print(f"30fps要件 (33.3ms): {'達成' if frontend_time < 33.3 else '未達成'}")
print(f"バックエンド合計: {sum(modules[m]['time_ms'] for m in ['Local BA', 'Loop Detection', 'Global Optimization', 'Map Update'])}ms (非同期)")

## 3. 階層的空間認識

現代の空間AIシステムは、複数の抽象レベルで環境を理解します：
- **低レベル**: ポイントクラウド、メッシュ、ボクセル（幾何）
- **中レベル**: 物体、平面、構造要素（セマンティック）
- **高レベル**: 部屋、フロア、建物（トポロジー）

この階層的表現により、タスクに応じた適切な粒度での空間推論が可能になります。

In [ ]:
# --- Hierarchical Spatial Perception ---

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Level 1: Low-level (point cloud)
ax = axes[0]
n_points = 200
# Simulate walls and objects as point clouds
wall_pts = np.vstack([
    np.column_stack([np.zeros(30), np.linspace(0, 10, 30)]),  # left wall
    np.column_stack([np.full(30, 10), np.linspace(0, 10, 30)]),  # right wall
    np.column_stack([np.linspace(0, 10, 30), np.zeros(30)]),  # bottom wall
    np.column_stack([np.linspace(0, 10, 30), np.full(30, 10)]),  # top wall
    np.column_stack([np.full(20, 5), np.linspace(0, 6, 20)]),  # inner wall
])
wall_pts += np.random.randn(*wall_pts.shape) * 0.1

obj_pts = np.vstack([
    np.random.randn(15, 2) * 0.5 + [2.5, 5],  # object 1
    np.random.randn(15, 2) * 0.4 + [7.5, 3],  # object 2
    np.random.randn(10, 2) * 0.3 + [7.5, 7],  # object 3
])

ax.scatter(wall_pts[:, 0], wall_pts[:, 1], c='gray', s=5, alpha=0.6)
ax.scatter(obj_pts[:, 0], obj_pts[:, 1], c='steelblue', s=10, alpha=0.6)
ax.set_title('Low-Level\n(Point Cloud)', fontsize=14, fontweight='bold')
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Level 2: Mid-level (objects + planes)
ax = axes[1]
# Walls as lines
ax.plot([0, 0, 10, 10, 0], [0, 10, 10, 0, 0], 'k-', linewidth=2, label='Walls')
ax.plot([5, 5], [0, 6], 'k-', linewidth=2)

# Objects as bounding boxes
objects = [
    ([1.5, 4], [2, 2], 'Table', '#8B4513'),
    ([6.5, 2], [2, 2], 'Desk', '#DAA520'),
    ([6.5, 6], [1.5, 1.5], 'Chair', '#FF8C00'),
]
for pos, size, name, color in objects:
    rect = plt.Rectangle(pos, size[0], size[1], facecolor=color, alpha=0.5,
                         edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(pos[0] + size[0]/2, pos[1] + size[1]/2, name,
            ha='center', va='center', fontsize=10, fontweight='bold')

# Door
ax.plot([5, 5], [6, 8], 'g-', linewidth=4, label='Door')

ax.set_title('Mid-Level\n(Objects + Structure)', fontsize=14, fontweight='bold')
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 11)
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Level 3: High-level (topological)
ax = axes[2]

# Rooms as regions
room1 = plt.Rectangle((0.5, 0.5), 4, 9, facecolor='#FFE4B5', alpha=0.5,
                       edgecolor='orange', linewidth=2)
room2 = plt.Rectangle((5.5, 0.5), 4, 9, facecolor='#E0E8FF', alpha=0.5,
                       edgecolor='blue', linewidth=2)
ax.add_patch(room1)
ax.add_patch(room2)

# Room labels
ax.text(2.5, 5, 'Room A\n(Living Room)', ha='center', va='center',
        fontsize=12, fontweight='bold', color='darkorange')
ax.text(7.5, 5, 'Room B\n(Office)', ha='center', va='center',
        fontsize=12, fontweight='bold', color='darkblue')

# Topological graph
ax.plot([2.5, 7.5], [8, 8], 'k-', linewidth=3)
ax.scatter([2.5, 7.5], [8, 8], s=200, c=['orange', 'blue'], zorder=5, edgecolors='black')
ax.scatter([5], [8], s=100, c='green', zorder=5, marker='D', edgecolors='black')
ax.text(5, 8.5, 'Door', ha='center', fontsize=10)

# Object counts
ax.text(2.5, 2, 'Contains:\n- 1 Table\n- 0 Chairs', ha='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.text(7.5, 2, 'Contains:\n- 1 Desk\n- 1 Chair', ha='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_title('High-Level\n(Topological / Rooms)', fontsize=14, fontweight='bold')
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hierarchical_perception.png', dpi=100, bbox_inches='tight')
plt.show()
print("同じ環境を3つの抽象レベルで表現: ポイントクラウド → 物体/構造 → 部屋/トポロジー")

## 演習問題

1. **クリティカルパス分析**: DAGに新しいモジュール（例：物体検出 12ms）を追加した場合、フロントエンドのレイテンシはどう変化しますか？GPU並列化は有効ですか？
2. **スケーラビリティ**: 地図のサイズが10倍になった場合、各モジュールの計算時間はどうスケールしますか？O(N), O(N log N), O(N^2) のどれですか？
3. **リソース配分**: CPU 4コア + GPU 1枚の制約下で、パイプラインのスレッド配分を最適化してください。

## まとめ

- 現代のSLAMシステムは**計算DAG**として構成され、フロントエンド（リアルタイム）とバックエンド（非同期最適化）が並列動作する
- **レイテンシ分析**により、ボトルネックの特定とリアルタイム性の確保が可能。GPU並列化やマルチスレッド化が鍵
- **階層的空間認識**は、ポイントクラウドから部屋レベルのトポロジーまで、複数の抽象度でシーンを理解する
- システム設計では、精度とレイテンシのトレードオフ、メモリ効率、ハードウェア制約を総合的に考慮する必要がある
- 将来の空間AIシステムは、幾何・セマンティック・言語を統合した**統一的なシーン表現**に向かっている